# 14 -- Version 1.0 Final Evaluation Review (READ-ONLY)

> **WATERMARK: FINAL-EVALUATION REVIEW NOTEBOOK -- NOT A DEVELOPMENT NOTEBOOK.**
> This notebook reviews the output of the single, sealed, already-completed
> Version 1.0 final evaluation (`evaluation/run_v1_final_evaluation.py`). It is
> the only place in this repository that ever displays real 2025 results.

Versions 0.2-0.12 are FROZEN, and the Version 1.0 final evaluation itself is
SEALED once it runs -- see CLAUDE.md "The one narrow exception: the sealed
Version 1.0 final evaluation" and `evaluation/run_v1_final_evaluation.py`'s
module docstring for the full policy.

## What this notebook is NOT

- It never downloads anything (no network access of any kind).
- It never fits, recalibrates, or retrains any model.
- It never scores any play, batter, or season -- it only reads
  `outputs/final_evaluation/v1/v1_final_report.json` and the public score
  table/leaderboards already written by a completed, sealed run.
- It never writes or modifies any file on disk.
- It is not a second evaluation entry point -- if you find yourself wanting
  to "just recompute one number here to check," that is a sign to go back to
  `evaluation/run_v1_final_evaluation.py`'s sealing rules, not to do it here.

## What this notebook DOES do before showing anything

Before displaying a single substantive result, it recomputes the report's
content hash and every frozen artifact's hash and compares them against the
sealed `seal.json` / `v1_final_report.json`'s own recorded manifest. If the
seal is missing, corrupted, or does not match, every results section below
prints a clear refusal instead of rendering -- see Section 1.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

sys.path.insert(0, str(Path("../evaluation").resolve()))

# Overridable via environment variable for offline structure/execution testing
# against a synthetic sealed fixture -- see tests/test_notebook_14_v1_review.py.
# Production default is the real, isolated Version 1.0 namespace.
ARTIFACTS_DIR = Path(
    __import__("os").environ.get(
        "CONTACT_LUCK_V1_ARTIFACTS_DIR", "../artifacts/final_evaluation/v1"
    )
)
OUTPUTS_DIR = Path(
    __import__("os").environ.get(
        "CONTACT_LUCK_V1_OUTPUTS_DIR", "../outputs/final_evaluation/v1"
    )
)
REPO_ROOT = Path(
    __import__("os").environ.get("CONTACT_LUCK_V1_REPO_ROOT", "..")
).resolve()

SEAL_PATH = ARTIFACTS_DIR / "seal.json"
REPORT_PATH = OUTPUTS_DIR / "v1_final_report.json"
PUBLIC_SCORE_PATH = OUTPUTS_DIR / "public_score_v1_final.json"
FAVORABLE_PATH = OUTPUTS_DIR / "favorable_leaderboard_v1_final.json"
UNFAVORABLE_PATH = OUTPUTS_DIR / "unfavorable_leaderboard_v1_final.json"

print(f"Reading seal from:   {SEAL_PATH}")
print(f"Reading report from: {REPORT_PATH}")


## 1. Seal verification -- MUST pass before anything substantive is shown

Recomputes `report_content_hash` (the exact same SHA-256-of-canonical-JSON
scheme `evaluation.run_v1_final_evaluation.hash_report` uses) and every
frozen artifact's SHA-256 (the same scheme
`evaluation.v1_final_evaluation_manifest.compute_file_sha256` uses), and
compares both against the sealed values. `SEAL_OK` gates every later
section -- if it is `False`, every remaining section refuses to render.


In [ ]:
from run_v1_final_evaluation import hash_report
from v1_final_evaluation_manifest import compute_file_sha256

SEAL_OK = False
SEAL_FAILURE_REASONS = []

seal = json.loads(SEAL_PATH.read_text()) if SEAL_PATH.exists() else None
report = json.loads(REPORT_PATH.read_text()) if REPORT_PATH.exists() else None

if seal is None:
    SEAL_FAILURE_REASONS.append(f"No seal file found at {SEAL_PATH}.")
if report is None:
    SEAL_FAILURE_REASONS.append(f"No final report found at {REPORT_PATH}.")

if seal is not None and report is not None:
    recomputed_report_hash = hash_report(report)
    if recomputed_report_hash != seal.get("report_content_hash"):
        SEAL_FAILURE_REASONS.append(
            "Report content hash does not match the sealed hash -- the report file has "
            "changed since it was sealed, or the seal belongs to a different report."
        )

    stored_artifact_hashes = report.get("manifest", {}).get("artifact_hashes", {})
    if not stored_artifact_hashes:
        SEAL_FAILURE_REASONS.append("Report's manifest has no artifact_hashes to verify.")
    else:
        mismatched = []
        for rel_path, expected_hash in stored_artifact_hashes.items():
            full_path = REPO_ROOT / rel_path
            if not full_path.is_file():
                mismatched.append((rel_path, "missing"))
                continue
            current_hash = compute_file_sha256(full_path)
            if current_hash != expected_hash:
                mismatched.append((rel_path, "changed"))
        if mismatched:
            SEAL_FAILURE_REASONS.append(
                f"{len(mismatched)} frozen artifact(s) no longer match the sealed hashes: "
                f"{mismatched}"
            )

    if not SEAL_FAILURE_REASONS:
        SEAL_OK = True

if SEAL_OK:
    print("SEAL VERIFIED -- report content hash and every frozen artifact hash match.")
    print(f"Sealed at: {seal['sealed_at']}  |  commit: {seal['repository_commit']}")
    if seal.get("defect_fix_of"):
        print(f"This is a defect-fix rerun of a prior seal from: {seal['defect_fix_of']}")
else:
    print("SEAL INVALID OR ABSENT -- refusing to render substantive results below.")
    for reason in SEAL_FAILURE_REASONS:
        print(f"  - {reason}")


## 2. Outcome classification

The predeclared outcome (`validated_as_frozen`, `validated_with_documented_
limitations`, or `final_evaluation_failed`) and the exact reasons the
classification rule (fixed BEFORE this evaluation ever ran -- see
`evaluation.v1_final_report.classify_final_evaluation_outcome`) produced.


In [ ]:
if SEAL_OK:
    outcome_block = report["outcome_classification"]
    banner = {
        "validated_as_frozen": "\N{WHITE HEAVY CHECK MARK} VALIDATED AS FROZEN",
        "validated_with_documented_limitations": (
            "\N{WARNING SIGN} VALIDATED WITH DOCUMENTED LIMITATIONS"
        ),
        "final_evaluation_failed": "\N{CROSS MARK} FINAL EVALUATION FAILED",
    }.get(outcome_block["outcome"], outcome_block["outcome"])
    print(banner)
    print()
    for reason in outcome_block["reasons"]:
        print(f"  - {reason}")
else:
    print("Skipped -- no valid seal.")


## 3. Frozen score definition and retrospective limitation

The exact, centralized public language (`mlb_luck_score.scoring.
public_labels`) -- never paraphrased here.


In [ ]:
if SEAL_OK:
    from mlb_luck_score.scoring.public_labels import (
        CONTACT_LUCK_RUNS_DEFINITION,
        CONTACT_LUCK_RUNS_LABEL,
        CONTACT_LUCK_RUNS_PER_100_DEFINITION,
        CONTACT_LUCK_RUNS_PER_100_LABEL,
        RETROSPECTIVE_LIMITATION,
    )

    print(f"{CONTACT_LUCK_RUNS_LABEL}: {CONTACT_LUCK_RUNS_DEFINITION}")
    print(f"{CONTACT_LUCK_RUNS_PER_100_LABEL}: {CONTACT_LUCK_RUNS_PER_100_DEFINITION}")
    print()
    print(RETROSPECTIVE_LIMITATION)
else:
    print("Skipped -- no valid seal.")


## 4. System-level checks (accounting, routing, reproducibility, contract)

Every required structural check from `evaluation.v1_system_evaluation`,
re-run on 2025 and recorded in the sealed report -- never recomputed here.


In [ ]:
if SEAL_OK:
    for name, result in report["system_checks"].items():
        print(f"--- {name} ---")
        print(json.dumps(result, indent=2, default=str))
        print()
else:
    print("Skipped -- no valid seal.")


## 5. Component metrics and calibration statuses

Contact, open-field outfield, infield, and advancement components, plus the
near-wall specialist -- **labeled explicitly** as provisional/frozen
regardless of how it performed on 2025 (Version 0.7 stays frozen; see
CLAUDE.md).


In [ ]:
if SEAL_OK:
    component_metrics = report["component_metrics"]
    near_wall = component_metrics.get("near_wall_specialist", {})
    print(
        f"near_wall_specialist status: {near_wall.get('status')!r} -- "
        f"{near_wall.get('status_reason')}"
    )
    print()
    for name in ("contact_model", "open_field_outfield_model", "infield_model", "advancement_model"):
        if name in component_metrics:
            print(f"--- {name} ---")
            print(json.dumps(component_metrics[name], indent=2, default=str)[:2000])
            print()
else:
    print("Skipped -- no valid seal.")


## 6. Distribution shift: 2025 vs. 2021-2024 development reference

Descriptive only (SMD/PSI) -- flags a shift, never corrects one, and never
feeds back into any frozen model or threshold.


In [ ]:
if SEAL_OK:
    shift = report["distribution_shift"]
    print(f"{len(shift['flags'])} flagged shift(s):")
    for flag in shift["flags"]:
        print(f"  - {flag}")
else:
    print("Skipped -- no valid seal.")


## 7. Qualification summary and interval-width distribution

Qualification counts, share qualified, and the distribution of 95%
game_pk-clustered bootstrap interval widths among qualified rows.


In [ ]:
if SEAL_OK:
    pub_summary = report["public_score_evaluation"]
    print(f"n_batter_seasons: {pub_summary['n_batter_seasons']}")
    print(f"n_qualified: {pub_summary['n_qualified']}")
    print(f"share_qualified: {pub_summary['share_qualified']}")
    print()
    print("Interval width distribution (qualified rows):")
    print(json.dumps(pub_summary["interval_width_distribution"], indent=2))
    print()
    print("Qualified interval interpretation shares:")
    print(json.dumps(pub_summary["qualified_interval_interpretation_shares"], indent=2))
else:
    print("Skipped -- no valid seal.")


## 8. Point estimates with intervals -- visible zero reference

Every displayed point estimate is shown WITH its interval, and a vertical
line at zero makes "interval crosses zero" immediately visible (never a
significance claim -- see `RETROSPECTIVE_LIMITATION` above).


In [ ]:
if SEAL_OK and PUBLIC_SCORE_PATH.exists():
    public_score_table = pd.read_json(PUBLIC_SCORE_PATH)
    qualified = public_score_table[public_score_table["qualification_status"] == "qualified"]
    sample = qualified.sort_values("contact_luck_runs_per_100").head(30)

    fig, ax = plt.subplots(figsize=(8, max(4, len(sample) * 0.3)))
    y = range(len(sample))
    ax.errorbar(
        sample["contact_luck_runs_per_100"],
        y,
        xerr=[
            sample["contact_luck_runs_per_100"] - sample["lower_95_interval"],
            sample["upper_95_interval"] - sample["contact_luck_runs_per_100"],
        ],
        fmt="o",
        capsize=3,
    )
    ax.axvline(0, color="black", linewidth=1, linestyle="--", label="zero (no reference)")
    ax.set_yticks(list(y))
    ax.set_yticklabels(sample["batter_id"].astype(str).tolist())
    ax.set_xlabel("Contact Luck Runs per 100 (point estimate, 95% interval)")
    ax.set_title("Sample of qualified batter-seasons -- interval overlap with zero is descriptive only")
    ax.legend()
    plt.tight_layout()
    plt.show()
elif SEAL_OK:
    print(f"No public score table found at {PUBLIC_SCORE_PATH}.")
else:
    print("Skipped -- no valid seal.")


## 9. Stability analyses

Split-half/odd-even reliability within 2025, interval width by sample size,
and (if a 2024 comparison table was supplied to the sealed run) the
2024-vs-2025 cross-season correlation among dual-qualified players. Low
correlation here is CONSISTENT with Contact Luck's retrospective purpose --
never a predictive-validity failure (see Section 3's limitation language).


In [ ]:
if SEAL_OK:
    stability = report["stability"]
    print(json.dumps(stability, indent=2, default=str)[:4000])
else:
    print("Skipped -- no valid seal.")


## 10. Leaderboards

"Most favorable realized luck" and "least favorable outcomes relative to
expectation" -- qualified rows only, competition ranking.


In [ ]:
if SEAL_OK and FAVORABLE_PATH.exists() and UNFAVORABLE_PATH.exists():
    favorable_leaderboard = pd.read_json(FAVORABLE_PATH)
    unfavorable_leaderboard = pd.read_json(UNFAVORABLE_PATH)
    print("Most favorable realized luck:")
    display(favorable_leaderboard.head(25))
    print("Least favorable outcomes relative to expectation:")
    display(unfavorable_leaderboard.head(25))
elif SEAL_OK:
    print("Leaderboard file(s) not found.")
else:
    print("Skipped -- no valid seal.")


## 11. Limitations

Every documented limitation carried into `validated_with_documented_
limitations` (if that was the outcome) -- distribution-shift flags and/or
provisional-component status, never silently dropped.


In [ ]:
if SEAL_OK:
    limitations = report.get("limitations", [])
    if limitations:
        for item in limitations:
            print(f"  - {item}")
    else:
        print("No limitations recorded.")
else:
    print("Skipped -- no valid seal.")


## 12. Ingestion provenance

Exact 2025 date range requested, source, retrieval timestamp, raw file
hashes, row/game counts, and join coverage -- recorded once, at ingestion
time, by the sealed run itself (never recomputed here).


In [ ]:
if SEAL_OK:
    print(json.dumps(report.get("ingestion_provenance", {}), indent=2, default=str)[:4000])
else:
    print("Skipped -- no valid seal.")


---

**End of Version 1.0 final-evaluation review.** This notebook rendered
results ONLY because the seal verification in Section 1 passed. Re-running
this notebook after the underlying report/artifacts change (a new commit, an
edited frozen file, a defect-fix reseal) will correctly show a refusal until
Section 1 is satisfied again.
